# 🛍️ Project: A Multi-User Conversational Product Recommender

![](https://i.imgur.com/7ZHpO6U.png)

## Learning Objectives
In this notebook, you will learn:
1. **Text-to-pandas filtering** - turn a natural-language request into a dataframe filter, with a guarded executor
2. **An LLM recommender chain** - recommend products from the filtered table
3. **A history-aware query rephraser** - combine a user's earlier requests into one standalone query
4. **Per-user persistent memory** - wrap the combined chain with SQLite-backed history per user

## Where this fits
An application of the module: the rephrasing idea from `06_conversational_rag_with_history.ipynb` and the
SQLite-backed sessions from `04_multi_user_sessions_and_persistence.ipynb`, applied to a product catalogue.

## Prerequisites
- `04_multi_user_sessions_and_persistence.ipynb` and `06_conversational_rag_with_history.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

> **⚠️ Legacy API**: `RunnableWithMessageHistory` is deprecated in `langchain-core` 1.3.3 and removed in 2.0; `SQLChatMessageHistory` comes from the sunset `langchain-community`. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](../notebooks/06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: writes filters, recommendations and rephrased queries
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📊 Part 1: The Product Catalogue

30 products across 10 categories, with price, rating and a free-text description.

In [ ]:
# ============================================================================
# PART 1: LOAD THE CATALOGUE
# ============================================================================
import pandas as pd

df = pd.read_csv("../assets/Ecommerce_Product_List.csv")

print(df.head().to_string())
print(f"\n{len(df)} products; categories: {sorted(df['Category'].unique())}")
print(f"Ratings: {sorted(df['Rating'].unique())}")

### Recommender workflow
- A **text-to-pandas chain** filters products on category, rating and price
- A **recommender chain** picks products from the filtered table using the descriptions
- A **query rephraser** merges a user's follow-ups ("want one larger than 10 inches") into one standalone query
- All three are combined and wrapped with **per-user memory**

![](https://i.imgur.com/qJIsErH.png)

---
## 🐼 Part 2: Text-to-Pandas Filter Chain

The LLM writes a pandas expression such as `df[(df['Category'] == 'Tablet') & (df['Rating'] >= 4)]`.
Running model-written code is risky. The executor below limits the damage — **only `df` in scope, no built-ins**,
a DataFrame result only, and the full table as the fallback — but an empty `__builtins__` is **not a sandbox**:
Python introspection can still reach dangerous objects. Never `eval` model output on untrusted input in
production; there, have the model return a structured filter (`with_structured_output`) and apply it in code.

In [ ]:
# ============================================================================
# PART 2: NATURAL LANGUAGE -> PANDAS FILTER -> TABLE
# ============================================================================
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, chain


@chain
def pandas_code_tool_executor(query: str) -> str:
    """Evaluate a model-written pandas expression over df only; fall back to the full table."""
    try:
        result_df = eval(query.strip().strip("`"), {"__builtins__": {}}, {"df": df})  # noqa: S307 - demo only: not a sandbox
    except Exception as error:  # a bad expression should not crash the conversation
        print(f"  (filter failed: {error!r} -> using the full table)")
        return df.to_markdown()
    if not isinstance(result_df, pd.DataFrame) or result_df.empty:
        return df.to_markdown()
    return result_df.to_markdown()


FILTER_PROMPT = """Given the following schema of a dataframe table, your task is to figure out the best pandas
query to filter the dataframe based on the user query which will be in natural language.

The schema is as follows:
#   Column        Non-Null Count  Dtype
0   Product_ID    30 non-null     object
1   Product_Name  30 non-null     object
2   Category      30 non-null     object
3   Price_USD     30 non-null     int64
4   Rating        30 non-null     float64
5   Description   30 non-null     object

Category has values: ['Laptop', 'Tablet', 'Smartphone', 'Smartwatch', 'Camera',
                      'Headphones', 'Mouse', 'Keyboard', 'Monitor', 'Charger']
Rating ranges from 1 - 5 in floats

Focus only on Category, Price_USD and Rating if the user mentions anything about these.
Do not make up column names, only use the above. If not, the pandas query should just return the full dataframe.
Remember the dataframe name is df. Return only the pandas query and nothing else, not as markdown.

User Query: {user_query}
Pandas Query:
"""

data_filter_chain = ChatPromptTemplate.from_template(FILTER_PROMPT) | llm | StrOutputParser() | pandas_code_tool_executor

print(data_filter_chain.invoke({"user_query": "looking for a tablet with > 10 inch display and at least 64GB storage"}))

Display size and storage are not columns, so the filter can only narrow by category — the table should list
the tablets. The recommender chain reads their descriptions for the rest.

---
## 🤝 Part 3: The Recommender Chain

In [ ]:
# ============================================================================
# PART 3: RECOMMEND FROM THE FILTERED TABLE
# ============================================================================
RECOMMEND_PROMPT = """Act as an expert retail product advisor. Given the following table of products, focus on the
product attributes and description in the table and, based on the user query below:

- Recommend the most appropriate products based on the query
- Each recommendation has product name, price, rating, description and a brief reason
- Do not make up products or recommend products not in the table
- If some specifications do not match, focus on the ones which match
- If nothing matches, recommend 5 random products from the table
- Do not generate anything else except the fields mentioned above

If the user query is just a generic query or greeting, respond appropriately without recommending products.

Product Table:
{product_table}

User Query:
{user_query}

Recommendation:
"""

recommend_chain = ChatPromptTemplate.from_template(RECOMMEND_PROMPT) | llm | StrOutputParser()

filter_then_recommend = {"user_query": itemgetter("user_query"), "product_table": data_filter_chain} | recommend_chain
print(filter_then_recommend.invoke({"user_query": "looking for a cheap laptop in the range of 500 - 1000"}))

---
## 🔁 Part 4: A History-Aware Query Rephraser

"want one which has display larger than 10 inches" is useless on its own. The rephraser reads the user's
recent history and returns a standalone query ("tablet with a display larger than 10 inches"). It only sees
the last `k` exchanges.

In [ ]:
# ============================================================================
# PART 4: REPHRASE THE LATEST REQUEST USING HISTORY
# ============================================================================
from langchain_core.prompts import MessagesPlaceholder

REPHRASE_SYS_PROMPT = """You are a retail product expert. Carefully analyze the following conversation history
and the current user query. Refer to the history and rephrase the current user query into a standalone query
which can be used without the history for making search queries. Rephrase only if needed.
Just return the query and do not answer it."""

rephrase_prompt = ChatPromptTemplate.from_messages([
    ("system", REPHRASE_SYS_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "Current User Query:\n{human_input}"),
])


def memory_buffer_window(messages, k: int = 10):
    """Keep only the last k user/AI exchanges (2k messages)."""
    return messages[-(2 * k):]


rephrase_query_chain = (
    RunnablePassthrough.assign(history=lambda x: memory_buffer_window(x["history"]))
    | rephrase_prompt
    | llm
    | StrOutputParser()
)
print("✅ Rephraser ready")

---
## 🧩 Part 5: Combine Everything with Per-User Memory

Rephrase → filter → recommend, wrapped with `RunnableWithMessageHistory` and one SQLite-backed history per user.
`RESET_MEMORY = True` starts from an empty database so the demo is repeatable; set it to `False` to keep
conversations across kernel restarts.

In [ ]:
# ============================================================================
# PART 5A: THE MULTI-USER CONVERSATIONAL RECOMMENDER
# ============================================================================
import os

from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from rich.console import Console
from rich.markdown import Markdown

DB_FILE = "product_agent_memory.db"
RESET_MEMORY = True
if RESET_MEMORY and os.path.exists(DB_FILE):
    os.remove(DB_FILE)


def get_session_history_db(session_id: str) -> SQLChatMessageHistory:
    return SQLChatMessageHistory(session_id=session_id, connection=f"sqlite:///{DB_FILE}")


recommender_chain = (
    {"human_input": itemgetter("human_input"), "history": itemgetter("history")}
    | {"user_query": rephrase_query_chain}
    | RunnablePassthrough.assign(product_table=data_filter_chain)
    | recommend_chain
)

conv_chain = RunnableWithMessageHistory(
    recommender_chain, get_session_history_db, input_messages_key="human_input", history_messages_key="history"
)
console = Console()


def chat_with_llm(prompt_text: str, session_id: str) -> None:
    console.print(f"[bold]{session_id}:[/bold] {prompt_text}")
    response = conv_chain.invoke({"human_input": prompt_text}, {"configurable": {"session_id": session_id}})
    console.print(Markdown(response))

In [ ]:
# ============================================================================
# PART 5B: USER 1 NARROWS DOWN A TABLET
# ============================================================================
chat_with_llm("looking for a tablet", "jim001")
chat_with_llm("want one which has display larger than 10 inches", "jim001")
chat_with_llm("need at least 128GB disk space", "jim001")

In [ ]:
# ============================================================================
# PART 5C: USER 2 HAS A SEPARATE CONVERSATION
# ============================================================================
chat_with_llm("I want a laptop with a high rating", "bond007")
chat_with_llm("want at least 16GB memory", "bond007")

Each follow-up for `jim001` should still be about tablets, and `bond007`'s about laptops: the rephraser rebuilt
each request from that user's own SQLite history.

---
## 📝 Summary

In this notebook, we learned:

### 1. Text-to-pandas filtering
- The model writes a filter; the executor runs it with only `df` in scope and falls back safely

### 2. An LLM recommender chain
- Recommendations come only from the filtered table, grounded in product descriptions

### 3. A history-aware query rephraser
- Follow-ups are merged with the last `k` exchanges into one standalone query

### 4. Per-user persistent memory
- `RunnableWithMessageHistory` + `SQLChatMessageHistory` keep each user's conversation separate

### Next Steps
- **`08_multi_user_research_agent_sql_memory.ipynb`**: give a tool-calling agent the same per-user memory